# Prompt Baseline

In [1]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys
import ctypes

try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/nccl/lib/libnccl.so.2")
except Exception:
    pass

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"
ARTIFACTS_DIR = f"{codebase_path}/artifacts"
print("💻 Local Lab Server Environment Loaded.")
cuda_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib"
nccl_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/nccl/lib"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cuda_link_path + ":" + nccl_link_path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


💻 Local Lab Server Environment Loaded.


## 1. Load Data & Base Model

In [2]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from src.inference.evaluate import run_evaluation

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

val_full = load_jsonl(f"{DATA_DIR}/val_full_info.jsonl")
val_struct = load_jsonl(f"{DATA_DIR}/val_structural.jsonl")
print(f"Loaded {len(val_full)} full-info validation samples and {len(val_struct)} structural validation samples.")

Loaded 2039 full-info validation samples and 2039 structural validation samples.


In [3]:
# === SELECT MODEL ===
MODEL_ID = "ibm-granite/granite-20b-functioncalling"

In [4]:
# === LOAD MODEL ===
COMPUTE_DTYPE = torch.bfloat16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

if MODEL_ID == "mistralai/Mistral-Nemo-Instruct-2407":
	tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
											trust_remote_code=True,
											fix_mistral_regex=True
											)
else:
	tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
											trust_remote_code=True,
											)
 
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=COMPUTE_DTYPE,
)
model.eval()
print("Model loaded successfully!")

Loading weights:   0%|          | 0/628 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded successfully!


## 2. Evaluate Baseline Full Information Prompt

In [5]:
# Evaluate on Full Info Dataset
acc_full, results_full = run_evaluation(
    model=model,
    tokenizer=tokenizer,
    dataset=val_full,
    training_strategy="baseline",
    prompt_format="FullInfo",
    model_name=MODEL_ID,
    output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating baseline - FullInfo:   0%|          | 0/2039 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Evaluating baseline - FullInfo:   1%|          | 20/2039 [00:12<20:40,  1.63it/s]

Evaluating baseline - FullInfo:   2%|▏         | 40/2039 [00:23<19:49,  1.68it/s]

...

Evaluating baseline - FullInfo:  97%|█████████▋| 1980/2039 [19:45<00:35,  1.67it/s]

Evaluating baseline - FullInfo:  98%|█████████▊| 2000/2039 [19:57<00:23,  1.68it/s]

Evaluating baseline - FullInfo:  99%|█████████▉| 2020/2039 [20:09<00:11,  1.67it/s]

Evaluating baseline - FullInfo: 100%|██████████| 2039/2039 [20:21<00:00,  1.67it/s]

\n--- Evaluation Results ---
Training Strategy: baseline
Prompt Format: FullInfo
Model: ibm-granite/granite-20b-functioncalling
Accuracy: 0.5768
Format Error Rate: 0.0010
Semantic Confusion: 0.3244
Option Bias (A): 0.1785
Latency: 1221.18 seconds
Detailed predictions saved to: /data220_2/emmy/mlbio/hw4/output/validation/validation_granite-20b-functioncalling_FullInfo_baseline.csv


## 3. Evaluate Baseline Structural-Only Prompt

In [6]:
# Evaluate on Structural Only Dataset
acc_struct, results_struct = run_evaluation(
    model=model,
    tokenizer=tokenizer,
    dataset=val_struct,
    training_strategy="baseline",
    prompt_format="structOnly",
    model_name=MODEL_ID,
    output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating baseline - structOnly:   0%|          | 0/2039 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Evaluating baseline - structOnly:   1%|          | 20/2039 [00:08<14:36,  2.30it/s]

Evaluating baseline - structOnly:   2%|▏         | 40/2039 [00:17<14:13,  2.34it/s]

...

Evaluating baseline - structOnly:  97%|█████████▋| 1980/2039 [14:15<00:25,  2.34it/s]

Evaluating baseline - structOnly:  98%|█████████▊| 2000/2039 [14:23<00:16,  2.33it/s]

Evaluating baseline - structOnly:  99%|█████████▉| 2020/2039 [14:32<00:08,  2.31it/s]

Evaluating baseline - structOnly: 100%|██████████| 2039/2039 [14:40<00:00,  2.31it/s]

\n--- Evaluation Results ---
Training Strategy: baseline
Prompt Format: structOnly
Model: ibm-granite/granite-20b-functioncalling
Accuracy: 0.6072
Format Error Rate: 0.0010
Semantic Confusion: 0.3465
Option Bias (A): 0.1555
Latency: 880.94 seconds
Detailed predictions saved to: /data220_2/emmy/mlbio/hw4/output/validation/validation_granite-20b-functioncalling_structOnly_baseline.csv
